## DOUAE

In [1]:
!git clone https://github.com/Dackss/ProjetIA.git


fatal: destination path 'ProjetIA' already exists and is not an empty directory.


# BESOIN 3 : ARCHITECTURE DE CLASSIFICATION ET SAUVEGARDE DES IMAGES PNG

### 1. Importation des bibliothèques et configuration

In [2]:

import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix

# Configuration des graphiques
sns.set_theme(style="whitegrid")

## 2. Préparation et nettoyage des données

In [3]:
# PRÉPAR & EXTRACT DES DONNÉES 
df = pd.read_csv("data/export_IA.csv", low_memory=False)

# CIBLE 
colonne_cible = 'implantation' 

# col REQUISES 
colonnes_features = ['puissance', 'nb_pdc', 'gratuit', 'prise_ccs', 'prise_type2','prise_chademo' ]

# Nettoyage des données
df_propre = df.dropna(subset=colonnes_features + [colonne_cible]).copy()

### 2.1 Encodage des variables catégorielles

In [4]:
colonnes_bool = ['gratuit', 'prise_ccs', 'prise_type2','prise_chademo']
for col in colonnes_bool:
    df_propre[col] = df_propre[col].astype(str).str.upper()
    df_propre[col] = df_propre[col].map({'TRUE': 1, 'FALSE': 0, '1': 1, '0': 0 , 'OUI': 1, 'NON': 0}).fillna(0).astype(int)

## 3. Analyse Exploratoire des Données (Visualisations)

In [5]:
print(" enregistrement des graphiques de justification...")

# Graphique 1 : Boxplot de la puissance par type d'implantation
plt.figure(figsize=(10, 5))
sns.boxplot(data=df_propre, x=colonne_cible, y='puissance', palette='Set2')
plt.title("Justification de la variable 'puissance' selon l'Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("figures/justification_puissance.png", dpi=300) # Sauvegarde en haute définition
plt.close() # Ferme la figure pour libérer la mémoire

# Graphique 2 : Proportion de prises CCS par type d'implantation
plt.figure(figsize=(10, 5))
sns.barplot(data=df_propre, x=colonne_cible, y='prise_ccs', errorbar=None, palette='Blues_r')
plt.title("Proportion de présence de Prises CCS par Implantation", fontsize=12, fontweight='bold')
plt.xticks(rotation=15)
plt.tight_layout()
plt.savefig("figures/proportion_prise_ccs.png", dpi=300)
plt.close()

 enregistrement des graphiques de justification...


C:\Users\HI\AppData\Local\Temp\ipykernel_20264\2048168663.py:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=df_propre, x=colonne_cible, y='puissance', palette='Set2')
C:\Users\HI\AppData\Local\Temp\ipykernel_20264\2048168663.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.barplot(data=df_propre, x=colonne_cible, y='prise_ccs', errorbar=None, palette='Blues_r')


# 4. Prétraitement des données (Preprocessing)

## 4.1 Séparation du jeu de données (Train/Test Split)

In [6]:
X = df_propre[colonnes_features].astype(float)
y = df_propre[colonne_cible]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

## 4.2 Normalisation des fonctionnalités et sauvegarde du Scaler

In [7]:

#  NORMALISATION ET ENREGISTREMENT DU SCALER
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

joblib.dump(scaler, 'fichier_pkl/scaler_pretraitement_b3.pkl')


['fichier_pkl/scaler_pretraitement_b3.pkl']

# 5. Entraînement du modèle et optimisation (GridSearchCV)

In [8]:
print("\nRecherche des meilleurs hyperparamètres (Régression Logistique)...")

param_grid = {
    'C': [0.1, 1.0, 10.0],
    'solver': ['lbfgs', 'saga'],
    'max_iter': [200]
}

modele_base = LogisticRegression(random_state=42)

grid_search = GridSearchCV(estimator=modele_base, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_train_scaled, y_train)

meilleur_modele = grid_search.best_estimator_
print(f"Meilleurs hyperparamètres : {grid_search.best_params_}\n")



Recherche des meilleurs hyperparamètres (Régression Logistique)...
Meilleurs hyperparamètres : {'C': 0.1, 'max_iter': 200, 'solver': 'lbfgs'}



# 6. Évaluation du modèle final

In [9]:
y_pred = meilleur_modele.predict(X_test_scaled)

print("         RAPPORT D'ÉVALUATION DE LA CLASSIFICATION         ")

print(classification_report(y_test, y_pred))

         RAPPORT D'ÉVALUATION DE LA CLASSIFICATION         
                                      precision    recall  f1-score   support

Parking privé réservé à la clientèle       1.00      0.32      0.48       307
        Parking privé à usage public       0.48      0.37      0.42      7499
                      Parking public       0.60      0.04      0.07      4110
 Station dédiée à la recharge rapide       0.58      0.38      0.46      1478
                              Voirie       0.51      0.86      0.64      9185

                            accuracy                           0.51     22579
                           macro avg       0.63      0.39      0.41     22579
                        weighted avg       0.53      0.51      0.45     22579



## 6.1 Matrice de confusion

In [10]:
matrice = confusion_matrix(y_test, y_pred)
classes = meilleur_modele.classes_

plt.figure(figsize=(8, 6))
sns.heatmap(matrice, annot=True, fmt='d', cmap='Blues', xticklabels=classes, yticklabels=classes)
plt.title("Matrice de Confusion - Implantation de la Station", fontsize=12, fontweight='bold')
plt.ylabel('Réalité Terrain')
plt.xlabel('Prédiction Modèle')
plt.tight_layout()
plt.savefig("figures/matrice_confusion.png", dpi=300) 
plt.close()


# 7. Exportation du modèle prédictif final

In [11]:
joblib.dump(meilleur_modele, 'fichier_pkl/modele_classification_b3.pkl')


['fichier_pkl/modele_classification_b3.pkl']